In [3]:
%pip -q install uv
!uv pip install --system -q "gradio>=5,<6" plotly scikit-learn pandas numpy openpyxl requests
!uv --version


uv 0.13.9 (x86_64-unknown-linux-gnu)


In [ ]:
#Importe de librerias
import os, re, html, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import gradio as gr
from IPython.display import display
warnings.filterwarnings('ignore')

#Diccionarios para el nombre del archivo que se requiere
PATRONES = {
    'matricula': 'MEN_MATRICULA_ESTADISTICA_ES_20261119',
    'desercion': 'articles-415244_recurso_18',
    'admitidos': 'articles-425153_recurso',
}
#Diccionario para la extension que debe tener cada nombre correspondiente del Diccionario PATRONES
EXTENSIONES = {'matricula': '.csv', 'desercion': '.xlsx', 'admitidos': '.xlsx'}

def encontrar_archivo(tipo):
    prefijo = PATRONES[tipo]
    extension = EXTENSIONES[tipo]
    #Candidatos guarda archivos que tengan el mismo nombre y extension
    candidatos = []
    for carpeta in [Path('/content'), Path.cwd(), Path('/mnt/data')]:
        if not carpeta.is_dir():
            continue
        for p in carpeta.glob(prefijo + '*' + extension):
            sufijo = p.stem[len(prefijo):]
            if sufijo == '' or re.fullmatch(r'\(\d+\)', sufijo):
                if p.is_file():
                    candidatos.append(p)
    if not candidatos:
        return None
    candidatos = list({str(p.resolve()): p for p in candidatos}.values())
    return max(candidatos, key=lambda p: p.stat().st_mtime)

#Identifica el tipo de archivo dentro de los candidatos, compara si tiene todos los filtros para ser utilizado
RUTAS = {tipo: encontrar_archivo(tipo) for tipo in PATRONES}
faltantes = [tipo for tipo, ruta in RUTAS.items() if ruta is None]
if faltantes:
    try:
        from google.colab import files
        print('Sube los archivos faltantes:', ', '.join(faltantes))
        files.upload()
    except ImportError:
        pass
    RUTAS = {tipo: encontrar_archivo(tipo) for tipo in PATRONES}

#Retorna un mensaje si el documento seleccionado no tiene las pautas esperadas dentro de los propios parametros prefijo y extension
faltantes = [tipo for tipo, ruta in RUTAS.items() if ruta is None]
if faltantes:
    raise FileNotFoundError('No se encontraron: ' + ', '.join(faltantes) +
                            '. Comprueba que los archivos estén en /content.')
print('Archivos encontrados:')
for tipo, ruta in RUTAS.items():
    print(f'  {tipo}: {ruta.name}')

Sube los archivos faltantes: matricula, desercion, admitidos
